[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C09_Reasoning_TTC_Course/01_cot_decomposition/01_cot_error_compounding.ipynb)

# 01 · CoT 与任务分解：误差复利的数学

<span style="background:#1f6feb;color:#fff;padding:2px 8px;border-radius:4px;font-size:12px">CPU</span> 纯 numpy/matplotlib 模拟推理器，自包含，不需要任何模型权重。

**本 notebook 你将完成：**

1. 用 $p(d)=\sigma(a-b\,d)$ 建模单步正确率，模拟"**一步到位** vs **分解为 $k$ 步**"的链式成功率 $[\sigma(a-bD/k)]^k$，看见**倒 U 形曲线**与最优步数 $k^*$；
2. 解析求出最优单步难度 $d^*$（只依赖能力 $a$、与总难度 $D$ 无关），验证 $k^*(D)\propto D$ —— 越难的任务越值得细分解；
3. 蒙特卡洛模拟与解析式对照验证；
4. 加入步骤间**传递噪声** $\varepsilon$ 与**自我纠错**概率 $r$，观察倒 U 曲线如何被压低、$k^*$ 如何左移又恢复；
5. 扫描能力参数 $a$，复现"**CoT 对小模型有害**、增益随规模出现"的交叉现象；
6. 4 道 ✏️ 练习。

参考：[Nye 2021] *Show Your Work: Scratchpads* (arXiv:2112.00114)、[Wei 2022] *Chain-of-Thought Prompting* (arXiv:2201.11903)、[Kojima 2022] *LLMs are Zero-Shot Reasoners* (arXiv:2205.11916)、[Zhou 2022] *Least-to-Most* (arXiv:2205.10625)、[Feng 2023] (arXiv:2305.15408)、[Merrill & Sabharwal 2023] (arXiv:2310.07923)。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def p_step(d, a, b):
    """单步正确率：做对一个难度为 d 的步骤的概率（IRT 两参数 logistic 形式）"""
    return sigmoid(a - b * d)

def p_direct(D, a, b):
    """一步到位：直接解总难度 D 的任务"""
    return p_step(D, a, b)

def p_chain(D, k, a, b):
    """分解为 k 步：每步难度 D/k，步骤独立、一错全错"""
    return p_step(D / k, a, b) ** k

A, B = 2.0, 0.6     # 默认能力 a / 难度敏感系数 b
D0 = 12.0           # 示例任务总难度

print(f"直接一步解 D={D0:g} : p = {p_direct(D0, A, B):.4f}")
for k in [2, 4, 8, 16, 64]:
    print(f"分解为 {k:2d} 步       : p = {p_chain(D0, k, A, B):.4f}")

上面的数字已经讲完了整个故事：一步到位几乎必败（0.6%），分成 8 步成功率提高 ~18 倍，但分成 64 步又掉回去 —— **分解不是越细越好**。

## 1 · 倒 U 形：分解收益 vs 误差复利

模型设定（与讲解第 3 节一致）：

- 单步正确率是单步难度 $d$ 的函数：$p(d)=\sigma(a-b\,d)$，$a$ 是模型**能力**，$b>0$ 是难度敏感度；
- 总难度 $D$ 的任务均匀分解为 $k$ 步，每步难度 $D/k$，步骤独立、**一错全错**：

$$ P_{\text{chain}}(k) = \big[\sigma(a - bD/k)\big]^k $$

两个极端都输：$k=1$ 时单步太难；$k\to\infty$ 时单步正确率封顶在 $\sigma(a)<1$，$P_{\text{chain}}\le\sigma(a)^k\to 0$ —— **误差复利（error compounding）**吃掉一切。中间必有最优分解粒度 $k^*$。

In [ ]:
ks = np.arange(1, 61)

plt.figure(figsize=(7.5, 4.5))
for D in [6.0, 12.0, 24.0]:
    curve = p_chain(D, ks, A, B)
    k_star = int(ks[np.argmax(curve)])
    line, = plt.plot(ks, curve, marker=".", ms=4, label=f"D={D:g}   k*={k_star}")
    plt.axvline(k_star, color=line.get_color(), ls=":", alpha=0.5)
plt.xlabel("number of steps k")
plt.ylabel("P_chain(k)")
plt.title("Decomposition vs error compounding: inverted-U in k")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print("观察：")
print("1) 每条曲线都是倒 U 形——先吃『单步变简单』的红利，后被『步数复利』反噬；")
print("2) 任务越难（D 越大），峰值越靠右：难任务值得更细的分解；")
print("3) D=24 时即使最优分解，成功率也只有 ~1%——分解不能替代能力，只能放大能力。")

## 2 · 解析最优：最优单步难度 $d^*$ 与 $k^*(D)\propto D$

对 $k$ 连续松弛，令 $f(k)=\log P_{\text{chain}}=k\log\sigma(u)$，其中 $u=a-bD/k$ 是"单步能力余量"。利用 $\tfrac{\mathrm d}{\mathrm du}\log\sigma(u)=1-\sigma(u)$ 与 $\tfrac{\mathrm du}{\mathrm dk}=bD/k^2$：

$$ f'(k) \;=\; \log\sigma(u) + k\,(1-\sigma(u))\,\frac{bD}{k^2}
        \;=\; \log\sigma(u) + (a-u)\,(1-\sigma(u)) \;\equiv\; g(u) $$

**$f'(k)$ 只通过 $u$ 依赖 $k$ 和 $D$**。于是一阶条件 $g(u^*)=0$ 解出一个与 $D$ 无关的 $u^*$，对应**最优单步难度** $d^*=(a-u^*)/b$，从而：

$$ k^*(D) \;=\; D/d^* \;\propto\; D $$

任务越难最优步数越多（线性增长），而"每步该多难"只由能力 $a$ 决定。下面：① 二分法解 $g(u^*)=0$；② 离散网格搜索验证 $k^*(D)$ 确实贴着直线 $D/d^*$ 走；③ 蒙特卡洛模拟对照解析式。

In [ ]:
# —— ① 二分法解 g(u*) = 0 ——
def g(u, a):
    s = sigmoid(u)
    return np.log(s) + (a - u) * (1 - s)

def solve_u_star(a):
    lo, hi = a - 50.0, a - 1e-9      # g(lo) ≈ a > 0，g(hi) = logσ(a) < 0
    for _ in range(100):
        mid = 0.5 * (lo + hi)
        if g(mid, a) > 0:
            lo = mid
        else:
            hi = mid
    return 0.5 * (lo + hi)

u_star = solve_u_star(A)
d_star = (A - u_star) / B
print(f"u* = {u_star:.4f}  ->  最优单步难度 d* = {d_star:.4f}（与 D 无关）")
print(f"预测 k*(D=12) = {D0/d_star:.2f}")

# —— ② 离散网格搜索 k*(D)，验证线性 ——
def optimal_k_grid(D, a, b, k_max=300):
    kk = np.arange(1, k_max + 1)
    return int(kk[np.argmax(p_chain(D, kk, a, b))])

Ds = np.arange(2, 41)
k_stars = np.array([optimal_k_grid(float(D), A, B) for D in Ds])
assert np.all(np.diff(k_stars) >= 0), "k*(D) 应随 D 单调不减"

plt.figure(figsize=(7, 4))
plt.plot(Ds, k_stars, marker="o", ms=4, label="grid-search k*(D)")
plt.plot(Ds, Ds / d_star, "--", label=f"analytic D/d* (d*={d_star:.2f})")
plt.xlabel("task difficulty D"); plt.ylabel("optimal #steps k*")
plt.title("Harder tasks deserve finer decomposition: k* ∝ D")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

# —— ③ 蒙特卡洛对照 ——
rng = np.random.default_rng(0)
def mc_chain(D, k, a, b, n=200_000):
    p = p_step(D / k, a, b)
    return float((rng.random((n, k)) < p).all(axis=1).mean())

print("\n蒙特卡洛 vs 解析式（n=200k）：")
for D, k in [(12.0, 1), (12.0, 6), (12.0, 20), (24.0, 12)]:
    mc, th = mc_chain(D, k, A, B), p_chain(D, k, A, B)
    print(f"  D={D:4.0f} k={k:2d}   MC={mc:.4f}   解析={th:.4f}   |Δ|={abs(mc-th):.4f}")
    assert abs(mc - th) < 5e-3
print("解析式与模拟一致 ✓")

## 3 · 传递噪声 $\varepsilon$ 与自我纠错 $r$

真实推理链的步骤不是干净的对/错：中间步骤会引入**被后续继承、放大的错误**（抄错中间结果、引用错结论）。最小扩展 —— 每步除了以 $p(D/k)$ 做对当步，还以概率 $\varepsilon$ 引入一个传递性错误，该错误以概率 $r$ 被后续察觉并纠正，则每步有效成功率与链式成功率为：

$$ q = p(D/k)\cdot\big(1-\varepsilon(1-r)\big), \qquad P_{\text{chain}} = q^k $$

定性预测：$\varepsilon$ 给每一步加一笔**与难度无关的固定税**，整条倒 U 曲线被压低、且峰值左移（步数越多暴露面越大，最优策略退向"少而大"的步子）；$r$ 把税退回去。这一对参数预告了课程两条主线：**process reward model**（模块 06）压 $\varepsilon(1-r)$；**self-correction 的真实 $r$ 有多大**（模块 07：没有外部反馈时常常 $\approx 0$）。

In [ ]:
def p_chain_noisy(D, k, a, b, eps, r):
    """带传递噪声 eps 与自我纠错 r 的链式成功率"""
    q = p_step(D / k, a, b) * (1 - eps * (1 - r))
    return q ** k

ks = np.arange(1, 61)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

for eps in [0.0, 0.02, 0.05, 0.10]:           # 左：噪声增大，曲线被压低、峰值左移
    c = p_chain_noisy(D0, ks, A, B, eps, 0.0)
    axes[0].plot(ks, c, marker=".", ms=3, label=f"eps={eps:g}  k*={int(ks[np.argmax(c)])}")
axes[0].set_title("transmission noise (r=0)")

for r in [0.0, 0.5, 0.9, 1.0]:                # 右：固定 eps=0.05，纠错把曲线抬回去
    c = p_chain_noisy(D0, ks, A, B, 0.05, r)
    axes[1].plot(ks, c, marker=".", ms=3, label=f"r={r:g}  k*={int(ks[np.argmax(c)])}")
axes[1].set_title("self-correction (eps=0.05)")

for ax in axes:
    ax.set_xlabel("number of steps k"); ax.legend(); ax.grid(alpha=0.3)
axes[0].set_ylabel("P_chain")
plt.tight_layout(); plt.show()

base = p_chain_noisy(D0, ks, A, B, 0.0, 0.0)
print(f"无噪声      : k*={int(ks[np.argmax(base)])}, 峰值={base.max():.4f}")
hurt = p_chain_noisy(D0, ks, A, B, 0.10, 0.0)
print(f"eps=0.10    : k*={int(ks[np.argmax(hurt)])}, 峰值={hurt.max():.4f}  <- 被压低且左移")
heal = p_chain_noisy(D0, ks, A, B, 0.10, 0.9)
print(f"eps=0.10,r=0.9: k*={int(ks[np.argmax(heal)])}, 峰值={heal.max():.4f}  <- 纠错近乎恢复")

## 4 · 规模效应：为什么 CoT 对小模型有害

[Wei 2022] 的实证：CoT 增益在小模型上为零甚至为负，跨过某个规模才出现 —— 经典的"涌现"曲线。误差分解模型给出一个**连续**机制：定义

$$ \mathrm{gain}(a) \;=\; \max_{k\ge 2} P_{\text{chain}}(k;a) \;-\; p_{\text{direct}}(a) $$

在 $a$ 很小的渐近区 $\sigma(z)\approx e^{z}$，于是 $P_{\text{chain}}(k)\approx e^{ka-bD}$ 而 $p_{\text{direct}}\approx e^{a-bD}$：**分解把能力惩罚 $a$ 付了 $k$ 次**（$ka < a$ 当 $a<0$），增益为负；$a$ 大时单步进入饱和区，复利温和而降难收益巨大，增益为正。增益曲线在中间穿过零点 —— 底层量平滑增长，表观指标却像相变（与 [Schaeffer 2023] "涌现是度量海市蜃楼"之争同构）。

In [ ]:
def best_chain(a, D=D0, b=B, k_max=200):
    kk = np.arange(2, k_max + 1)
    return float(np.max(p_chain(D, kk, a, b)))

a_grid = np.linspace(-2.0, 8.0, 201)
direct = p_direct(D0, a_grid, B)
chain  = np.array([best_chain(a) for a in a_grid])
gains  = chain - direct
a_cross = float(a_grid[np.argmax(gains > 0)])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].semilogy(a_grid, direct, label="direct (k=1)")
axes[0].semilogy(a_grid, chain, label="best chain (k>=2)")
axes[0].axvline(a_cross, color="gray", ls=":")
axes[0].set_xlabel("ability a"); axes[0].set_ylabel("success prob (log)")
axes[0].set_title("weak models: chain loses to direct"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(a_grid, gains, color="#d33")
axes[1].axhline(0, color="gray", lw=1)
axes[1].axvline(a_cross, color="gray", ls=":")
axes[1].set_xlabel("ability a"); axes[1].set_ylabel("CoT gain")
axes[1].set_title(f"gain(a) crosses zero at a≈{a_cross:.2f} -> looks 'emergent'")
axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"gain(a=-1) = {gains[np.argmin(abs(a_grid - (-1)))]:+.2e}   (分解有害)")
print(f"gain(a=+4) = {gains[np.argmin(abs(a_grid - 4))]:+.4f}     (分解大赚)")
print(f"零点 a ≈ {a_cross:.2f}：能力连续增长，CoT 增益却像突然『涌现』——")
print("机制只是 max_k 与乘性链这两个非线性观测函数。")

---
## ✏️ 练习 1：实现 `p_chain_ex` 并验证倒 U 形

不翻上文，自己实现链式成功率的解析式：`p_chain_ex(D, k, a, b)` 返回 $[\sigma(a-bD/k)]^k$，要求 `k` 既可以是标量也可以是 `np.ndarray`（逐元素计算）。

**提示**：numpy 广播天然支持，1–3 行即可；$k=1$ 时应精确退化为一步到位的 $\sigma(a-bD)$；自测会检查曲线在 $k\in[1,100]$ 上**单调升→单调降**（内点最大值，即倒 U 形）。

In [ ]:
def p_chain_ex(D, k, a, b):
    # TODO: 返回 [sigmoid(a - b*D/k)]**k，支持 k 为标量或 np.ndarray
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
kk = np.arange(1, 101)
curve = p_chain_ex(12.0, kk, 2.0, 0.6)

assert abs(p_chain_ex(12.0, 1, 2.0, 0.6) - 1/(1 + np.exp(-(2.0 - 0.6*12.0)))) < 1e-12  # k=1 退化
assert np.allclose(curve, p_chain(12.0, kk, 2.0, 0.6))          # 与正文实现一致
i = int(np.argmax(curve))
assert 0 < i < len(kk) - 1, "最大值应是内点（倒 U 形），不应在 k=1 或 k=100 处"
diffs = np.diff(curve)
assert np.all(diffs[:i] > 0) and np.all(diffs[i:] < 0), "应先单调升后单调降"
assert kk[i] == 6                                               # D=12, a=2, b=0.6 时 k*=6
print("✅ 练习 1 通过")

## ✏️ 练习 2：实现 `optimal_k` 并验证 $k^*(D)$ 单调不减

实现 `optimal_k(D, a, b, k_max=300)`：在 $k\in\{1,\dots,k_{\max}\}$ 上网格搜索，返回使 $P_{\text{chain}}$ 最大的整数 $k^*$。

**提示**：`np.arange(1, k_max+1)` + `np.argmax`，3 行左右；注意返回的是 **k 的取值**而不是数组下标（差 1）；简单任务（小 $D$）的答案应是 1 —— 不值得分解。

In [ ]:
def optimal_k(D, a, b, k_max=300):
    # TODO: 网格搜索 k = 1..k_max，返回使 p_chain(D, k, a, b) 最大的整数 k
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
assert optimal_k(2.0, 2.0, 0.6) == 1                    # 简单任务：别分解
assert optimal_k(12.0, 2.0, 0.6) == 6
assert abs(optimal_k(24.0, 2.0, 0.6) - 12) <= 1
assert abs(optimal_k(36.0, 2.0, 0.6) - 3 * optimal_k(12.0, 2.0, 0.6)) <= 2   # 近似线性
k_list = [optimal_k(float(D), 2.0, 0.6) for D in np.arange(2, 31, 2)]
assert all(k_list[i] <= k_list[i+1] for i in range(len(k_list)-1)), "k*(D) 应单调不减"
print("✅ 练习 2 通过")

## ✏️ 练习 3：带自我纠错的链 —— 递推式 vs 蒙特卡洛

把自我纠错建模成"**失败后重试一次**"：每一步先以 $p=\sigma(a-bD/k)$ 尝试；若失败，以概率 $r$ 察觉错误并重试一次（重试成功率仍为 $p$）；重试再失败则整链失败。实现两个函数并验证一致：

1. `p_chain_sc(D, k, a, b, r)`：解析式 —— 先写出单步有效成功率 $q$，再返回 $q^k$；
2. `mc_chain_sc(D, k, a, b, r, n, rng)`：蒙特卡洛 —— 逐步模拟上述过程 $n$ 条链，返回成功比例。

**提示**：$q = p + (1-p)\,r\,p$（首试成功，或 失败→察觉→重试成功）；$r{=}0$ 应退化为 `p_chain`，$r{=}1$ 时 $q = 1-(1-p)^2$。蒙特卡洛用布尔矩阵向量化：`first | (~first & detect & retry)`，每个矩阵形状 `(n, k)`，再 `.all(axis=1).mean()`。两函数合计 ~10 行。

In [ ]:
def p_chain_sc(D, k, a, b, r):
    # TODO: q = p + (1-p)*r*p，返回 q**k
    raise NotImplementedError

def mc_chain_sc(D, k, a, b, r, n=200_000, rng=None):
    # TODO: 模拟 n 条链：每步 首试成功 | (首试失败 & 察觉(r) & 重试成功)
    #       全部 k 步成功才算链成功，返回成功比例（float）
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
p6 = p_step(12.0/6, 2.0, 0.6)
assert np.isclose(p_chain_sc(12.0, 6, 2.0, 0.6, 0.0), p_chain(12.0, 6, 2.0, 0.6))   # r=0 退化
assert np.isclose(p_chain_sc(12.0, 6, 2.0, 0.6, 1.0), (1 - (1-p6)**2) ** 6)         # r=1 闭式
vals = [p_chain_sc(12.0, 6, 2.0, 0.6, r) for r in (0.0, 0.3, 0.6, 0.9)]
assert all(vals[i] < vals[i+1] for i in range(3)), "成功率应随 r 单调上升"
rng_t = np.random.default_rng(0)
for r in (0.0, 0.5, 0.9):
    mc = mc_chain_sc(12.0, 6, 2.0, 0.6, r, n=200_000, rng=rng_t)
    th = p_chain_sc(12.0, 6, 2.0, 0.6, r)
    assert abs(mc - th) < 0.01, f"r={r}: MC={mc:.4f} 与解析 {th:.4f} 不符"
print("✅ 练习 3 通过")

## ✏️ 练习 4：实现 CoT 增益 `gain_ex(a)` 并验证能力交叉

实现 `gain_ex(a, D, b, k_max=200)` $= \max_{k\ge 2} P_{\text{chain}}(k;a) - p_{\text{direct}}(a)$，并验证：**小 $a$ 时增益 $\le 0$（CoT 有害），大 $a$ 时增益 $> 0$，中间存在一次符号交叉**。

**提示**：注意 $k$ 从 **2** 开始（$k{=}1$ 就是 direct 本身）；复用 `p_chain` / `p_direct`，3 行左右；弱模型区的增益绝对值非常小（~1e-4），断言只看符号。

In [ ]:
def gain_ex(a, D=12.0, b=0.6, k_max=200):
    # TODO: max_{k=2..k_max} p_chain(D, k, a, b) - p_direct(D, a, b)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
assert gain_ex(-2.0) < 0 and gain_ex(-1.0) < 0 and gain_ex(0.0) < 0   # 弱模型：分解有害
assert gain_ex(2.0) > 0 and gain_ex(4.0) > 0.5                        # 强模型：分解大赚
assert gain_ex(6.0) > gain_ex(4.0)
gs = np.array([gain_ex(float(a)) for a in np.linspace(-2.0, 6.0, 81)])
idx = int(np.argmax(gs > 0))
assert idx > 0 and np.all(gs[:idx] <= 0) and np.all(gs[idx:] > 0), "应恰好交叉一次"
print(f"✅ 练习 4 通过（交叉点 a ≈ {np.linspace(-2.0, 6.0, 81)[idx]:.2f}）")

---
## 📖 参考答案

In [ ]:
# 练习 1 参考答案（先自己做，再对照）
def p_chain_ex(D, k, a, b):
    return sigmoid(a - b * D / np.asarray(k, dtype=float)) ** np.asarray(k, dtype=float)

In [ ]:
# 练习 2 参考答案（先自己做，再对照）
def optimal_k(D, a, b, k_max=300):
    kk = np.arange(1, k_max + 1)
    return int(kk[np.argmax(p_chain(D, kk, a, b))])

In [ ]:
# 练习 3 参考答案（先自己做，再对照）
def p_chain_sc(D, k, a, b, r):
    p = p_step(D / k, a, b)
    q = p + (1 - p) * r * p          # 首试成功，或 失败 -> 察觉(r) -> 重试成功
    return q ** k

def mc_chain_sc(D, k, a, b, r, n=200_000, rng=None):
    rng = rng or np.random.default_rng(0)
    p = p_step(D / k, a, b)
    first  = rng.random((n, k)) < p
    detect = rng.random((n, k)) < r
    retry  = rng.random((n, k)) < p
    step_ok = first | (~first & detect & retry)
    return float(step_ok.all(axis=1).mean())

In [ ]:
# 练习 4 参考答案（先自己做，再对照）
def gain_ex(a, D=12.0, b=0.6, k_max=200):
    kk = np.arange(2, k_max + 1)
    return float(np.max(p_chain(D, kk, a, b)) - p_direct(D, a, b))

---
## 小结

- **倒 U 形**：分解的收益（单步变简单）与代价（误差复利 $q^k$）相互对抗，$P_{\text{chain}}(k)=[\sigma(a-bD/k)]^k$ 必有内点最优 $k^*$ —— 想得太少做不动，想得太碎错得多。
- **$k^*(D)=D/d^*\propto D$**，且最优单步难度 $d^*$ 只依赖能力 $a$：难题深想、易题速答的最简理论，也是 overthinking（超过 $k^*$ 纯付复利税）的最简模型。
- **噪声与纠错**：传递噪声 $\varepsilon$ 压低曲线并把 $k^*$ 左推，自我纠错 $r$ 抬回去 —— process reward（模块 06）与 self-correction（模块 07）就是在操纵这两个参数。
- **gain(a) 交叉**：弱模型把能力惩罚付 $k$ 次、分解必亏；强模型进入饱和区、分解大赚 —— 连续机制产生"CoT 涌现"的表观相变。评测报"模型无 CoT 能力"前，先想想交叉点。

**下一步 → 模块 02 · Test-Time Scaling Laws**：本章把"分解几步"当成自由参数，下一章把它变成预算 —— 固定 token 预算下，准确率如何随测试时计算量 scaling，以及为什么应该报告 accuracy@budget 曲线而非单点分数。

---
## 🎯 真实数据胶囊题：真实 GSM8K 步数上的误差复利

CoT 越长，单步小错累积成大错：最终正确率约 `p^k`（p=单步正确率，k=步数）。用真实 GSM8K 的步数分布，算给定单步正确率下的期望最终正确率。

> 本模块新增的**真实数据**练习：自包含、用真实 GSM8K 把本章方法跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, json, urllib.request, re
import numpy as np
CACHE=os.path.expanduser("~/.reasoning_ttc_data"); os.makedirs(CACHE,exist_ok=True)
def _f(url,fn):
    p=os.path.join(CACHE,fn)
    if not os.path.exists(p): urllib.request.urlretrieve(url,p)
    return p
def gsm8k(n=300):
    p=_f("https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/test.jsonl","gsm8k_test.jsonl")
    return [json.loads(l) for l in open(p).read().splitlines()[:n]]
def gold(a): return a.split("####")[-1].strip().replace(",","")
def steps(a): return max(1, a.count("<<"))   # 真实推理步数代理

rows=gsm8k(300)
ks=np.array([steps(r["answer"]) for r in rows])
print(f"真实步数分布: 均值={ks.mean():.1f} 最大={ks.max()}")

**练习**：实现 `final_accuracy(p_step, ks)`：每题最终正确概率 `p_step**k`，返回所有题平均。验证：p_step 高时随步数衰减慢，p_step 低时衰减快。

In [ ]:
def final_accuracy(p_step, ks):
    # TODO: mean(p_step ** ks)
    raise NotImplementedError


In [ ]:
# 自测
a99=final_accuracy(0.99, ks); a90=final_accuracy(0.90, ks)
assert a99 > a90, "单步越准最终越准"
assert 0 < a90 < a99 <= 1
# 步数越多衰减越狠：长题正确率 < 短题
long_k=ks[ks>=np.median(ks)]; short_k=ks[ks<np.median(ks)]
assert final_accuracy(0.9, long_k) < final_accuracy(0.9, short_k)
print(f"单步0.99 -> 最终{a99:.2f}; 单步0.90 -> 最终{a90:.2f} ✓ (误差复利)")


### 📖 参考答案

In [ ]:
def final_accuracy(p_step, ks):
    return float(np.mean(p_step ** np.asarray(ks)))
print("✓ p^k 解释了为什么长链推理需要极高的单步可靠性")